In [0]:
%fs

dbutils.fs provides utilities for working with FileSystems. Most methods in
this package can take either a DBFS path (e.g., "/foo" or "dbfs:/foo"), or
another FileSystem URI.

For more info about a method, use dbutils.fs.help("methodName") .

In notebooks, you can also use the %fs shorthand to access DBFS. The %fs shorthand maps
straightforwardly onto dbutils calls. For example, "%fs head --maxBytes=10000 /file/path"
translates into "dbutils.fs.head("/file/path", maxBytes = 10000)".
 cp(from: String, to: String, recurse: boolean): boolean -> Copies a file or directory, possibly across FileSystems head(file: String, maxBytes: int): String -> Returns up to the first 'maxBytes' bytes of the given file as a String encoded in UTF-8 ls(dir: String): Seq -> Lists the contents of a directory mkdirs(dir: String): boolean -> Creates the given directory if it does not exist, also creating any necessary parent directories mount(source: String, mountPoint: String, encryptionType: String, owner: String, extraConfigs: Map): boolean -> Mounts the given source directory into DBFS at the given mount point mounts: Seq -> Displays information about what is mounted within DBFS mv(from: String, to: String, recurse: boolean): boolean -> Moves a file or directory, possibly across FileSystems put(file: String, contents: String, overwrite: boolean): boolean -> Writes the given String out to a file, encoded in UTF-8 refreshMounts: boolean -> Forces all machines in this cluster to refresh their mount cache, ensuring they receive the most recent information rm(dir: String, recurse: boolean): boolean -> Removes a file or directory unmount(mountPoint: String): boolean -> Deletes a DBFS mount point updateMount(source: String, mountPoint: String, encryptionType: String, owner: String, extraConfigs: Map): boolean -> Similar to mount(), but updates an existing mount point (if present) instead of creating a new one

In [0]:
df_master = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("dbfs:/Workspace/Users/tushardatabrickss/Data/cleaned_superstore.csv")


In [0]:
df_master = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/Workspace/tushardatabrickss/Data/cleaned_superstore.csv")

df_master.show(5)


+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|total_amount|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|    

In [0]:
df_master_clean = df_master.dropna().dropDuplicates()
df_master_clean.show(5)


+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+----------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|      City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount| Profit|total_amount|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+----------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+------------+
|    13|CA-2017-114412|2017-04-15|2017-04-20|Standard Class|   AA-10480|     Andrew Allen|   Consumer|United States|   Concord|North Carolina|      28027|  South|OFF-PA-100

In [0]:
# Save cleaned master as transactional Delta table
df_master_clean.write.format("delta").mode("overwrite").saveAsTable("customer_master_txn")

# Create a new customer profile table with extended schema
df_customer_profile = df_master_clean.select("Customer_ID", "Customer_Name") \
    .withColumn("Email", lit(None)) \
    .withColumn("Customer_Type", lit(None)) \
    .withColumn("Registration_Date", lit(None))

df_customer_profile.write.format("delta").mode("overwrite").saveAsTable("customer_master_profile")


In [0]:
df_incremental = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/Workspace/tushardatabrickss/Data/customer_incremental.csv")

# Apply same column renaming
df_incremental = df_incremental.toDF(*[c.replace(" ", "_") for c in df_incremental.columns])


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "customer_master")

deltaTable.alias("master").merge(
    df_incremental.alias("updates"),
    "master.Customer_ID = updates.Customer_ID"
).whenMatchedUpdate(set={
    "Customer_Name": "updates.Customer_Name",
    "Email": "updates.Email",
    "Customer_Type": "updates.Customer_Type",
    "Registration_Date": "updates.Registration_Date"
}).whenNotMatchedInsert(values={
    "Customer_ID": "updates.Customer_ID",
    "Customer_Name": "updates.Customer_Name",
    "Email": "updates.Email",
    "Customer_Type": "updates.Customer_Type",
    "Registration_Date": "updates.Registration_Date"
}).execute()


In [0]:
final_df = spark.table("customer_master_profile")

print("Row count:", final_df.count())
print("Unique rows:", final_df.dropDuplicates().count())
final_df.show(10)


Row count: 9994
Unique rows: 793
+-----------+-----------------+-----+-------------+-----------------+
|Customer_ID|    Customer_Name|Email|Customer_Type|Registration_Date|
+-----------+-----------------+-----+-------------+-----------------+
|   AA-10480|     Andrew Allen| NULL|         NULL|             NULL|
|   SN-20710|     Steve Nguyen| NULL|         NULL|             NULL|
|   DW-13585|   Dorothy Wardle| NULL|         NULL|             NULL|
|   LC-16885|   Lena Creighton| NULL|         NULL|             NULL|
|   TW-21025|Tamara Willingham| NULL|         NULL|             NULL|
|   RD-19900|      Ruben Dartt| NULL|         NULL|             NULL|
|   JK-15640|         Jim Kriz| NULL|         NULL|             NULL|
|   JB-15925|   Joni Blumstein| NULL|         NULL|             NULL|
|   KL-16645|     Ken Lonsdale| NULL|         NULL|             NULL|
|   DW-13480|    Dianna Wilson| NULL|         NULL|             NULL|
+-----------+-----------------+-----+-------------+------

In [0]:
%sql

SELECT * FROM customer_master_profile;


Customer_ID,Customer_Name,Email,Customer_Type,Registration_Date
AA-10480,Andrew Allen,null,null,null
SN-20710,Steve Nguyen,null,null,null
DW-13585,Dorothy Wardle,null,null,null
LC-16885,Lena Creighton,null,null,null
TW-21025,Tamara Willingham,null,null,null
RD-19900,Ruben Dartt,null,null,null
JK-15640,Jim Kriz,null,null,null
JB-15925,Joni Blumstein,null,null,null
KL-16645,Ken Lonsdale,null,null,null
DW-13480,Dianna Wilson,null,null,null


In [0]:
%sql
SELECT txn.Customer_ID,
       txn.Customer_Name,
       profile.Email,
       profile.Customer_Type,
       profile.Registration_Date,
       txn.Sales,
       txn.Profit
FROM customer_master_txn txn
LEFT JOIN customer_master_profile profile
ON txn.Customer_ID = profile.Customer_ID;


Customer_ID,Customer_Name,Email,Customer_Type,Registration_Date,Sales,Profit
AA-10480,Andrew Allen,null,null,null,15.552,5.4432
SN-20710,Steve Nguyen,null,null,null,113.328,35.415
DW-13585,Dorothy Wardle,null,null,null,78.304,29.364
LC-16885,Lena Creighton,null,null,null,65.88,18.4464
TW-21025,Tamara Willingham,null,null,null,203.184,15.2388
RD-19900,Ruben Dartt,null,null,null,28.4,13.348
JK-15640,Jim Kriz,null,null,null,3.28,1.4104
JB-15925,Joni Blumstein,null,null,null,22.704,8.2302
KL-16645,Ken Lonsdale,null,null,null,419.68,-356.728
DW-13480,Dianna Wilson,null,null,null,47.88,23.94


In [0]:
%sql
SELECT Customer_Type, COUNT(*) AS Customer_Count
FROM customer_master_profile
GROUP BY Customer_Type;


Customer_Type,Customer_Count
null,9994


In [0]:
%sql
SELECT txn.Segment,
       SUM(TRY_CAST(txn.Sales AS DOUBLE)) AS Total_Sales,
       SUM(TRY_CAST(txn.Profit AS DOUBLE)) AS Total_Profit
FROM customer_master_txn txn
GROUP BY txn.Segment
ORDER BY Total_Sales DESC;


Segment,Total_Sales,Total_Profit
Consumer,1150166.1819999907,134443.8591999995
Corporate,696604.5138,91429.96490000012
Home Office,425679.1605000004,59833.77810000006


In [0]:
%sql
SELECT txn.Customer_ID, txn.Customer_Name, SUM(txn.Profit) AS Profit
FROM customer_master_txn txn
GROUP BY txn.Customer_ID, txn.Customer_Name
ORDER BY Profit DESC
LIMIT 10;


Customer_ID,Customer_Name,Profit
TC-20980,Tamara Chand,8964.4826
RB-19360,Raymond Buch,6976.0959
SC-20095,Sanjit Chand,5757.411899999999
HL-15040,Hunter Lopez,5622.4292
AB-10105,Adrian Barton,5438.9075
TA-21385,Tom Ashbrook,4703.788299999999
CM-12385,Christopher Martinez,3899.8904
KD-16495,Keith Dawkins,3038.6254
AR-10540,Andy Reiter,2884.6208
DR-12940,Daniel Raglin,2869.076


In [0]:
%sql
SELECT profile.Customer_Type,
       SUM(TRY_CAST(txn.Sales AS DOUBLE)) AS Sales,
       SUM(TRY_CAST(txn.Profit AS DOUBLE)) AS Profit
FROM customer_master_txn txn
LEFT JOIN customer_master_profile profile
ON txn.Customer_ID = profile.Customer_ID
GROUP BY profile.Customer_Type;


Customer_Type,Sales,Profit
null,3.548707902730629E7,4542029.870700177
